# MLST Additional Experiments Notebook

**Six priority experiments** to supplement the existing verification suite for MLST submission.

| # | Experiment | Paper anchor |
|---|---|---|
| P1 | σ_E vs k — pink / white / brownian noise, OF dashed baseline | §7 main results |
| P2 | SNR spectral density \|T(f)\|²/J(f) with PC1 overlay | §4.3 Theorem 1 visual |
| P3 | 3 × 3 PC cosine matrix across noise types | §4 PC interpretation |
| P4 | Timing-bias reduction: rank-1 vs rank-2 | §4 physical consequence |
| P5 | Residual χ² distribution as a function of k | §4.4 rank saturation |
| P6 | Bootstrap CIs on cosine scores | §4 PC interpretation |

Run cells top-to-bottom.  All experiments use the **simulation infrastructure**
from `experiments/notebook_support.py` — no real data required.


## 0 — Setup

In [1]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats

warnings.filterwarnings('ignore', category=RuntimeWarning)

# ── locate repo root ─────────────────────────────────────────────────────────
def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'QP_simulator').exists() and (p / 'experiments').exists():
            return p
    raise RuntimeError(f'Cannot locate repo root from {start}')

REPO = find_repo_root()
for _p in [REPO, REPO / 'src', REPO / 'QP_simulator', REPO / 'experiments', REPO / 'implementation']:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

FIG_DIR = REPO / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO)


Repo root: /Users/wongdowling/Documents/noise-weighted-subspace-reconstruction


In [2]:
from notebook_support import (
    CanonicalConfig,
    simulate_rank1_batch,
    simulate_controlled_family,
    fit_weighted_empca,
    exact_weighted_subspace,
    rankk_gls_coefficients,
    residual_energy_per_trace,
    compute_of_statistics,
    stationary_noise_generator,
    build_of_one_sided_weights,
    phase_align_basis,
)
from empca_equivalence_utils import weighted_cosine, weighted_inner

print('Imports OK.')


RuntimeError: Could not locate repo root from start path: /Users/wongdowling/Documents/noise-weighted-subspace-reconstruction/experiments/notebook_support.py

In [ ]:
mpl.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 11, 'axes.labelsize': 11,
    'figure.dpi': 150, 'lines.linewidth': 1.6,
    'axes.grid': True, 'grid.alpha': 0.3,
})
BLUE, ORANGE, GREEN, RED, PURPLE = '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'
NOISE_COLORS = {'pink': BLUE, 'white': ORANGE, 'brownian': GREEN}
NOISE_LABELS = {'pink': 'Pink (1/f)', 'white': 'White', 'brownian': 'Brownian (1/f²)'}

def save_fig(fig, name):
    p = FIG_DIR / name
    fig.savefig(p, bbox_inches='tight', dpi=150)
    print(f'  saved -> results/figures/{name}')

def normalize_weighted(v, w):
    n = np.sqrt(np.real(weighted_inner(v, v, w)))
    return v / n if n > 0 else v


In [ ]:
# ── canonical config & noise types ──────────────────────────────────────────
cfg = CanonicalConfig(
    seed=314159,
    sim_events_medium=300,   # enough for good statistics; increase if time allows
    sim_events_large=300,
    empirical_rank_max=8,
).validate()

NOISE_TYPES = ['pink', 'white', 'brownian']
K_MAX       = cfg.empirical_rank_max
N_SEEDS     = 5          # seeds for error bands; increase to 8 for final paper run
N_EVENTS    = cfg.sim_events_large
NOISE_POWER = 1.0

print(f'Config: N_events={N_EVENTS}, K_max={K_MAX}, N_seeds={N_SEEDS}')
print(f'Noise types: {NOISE_TYPES}')


---
## P1 — σ_E vs k for Pink / White / Brownian Noise

*Paper §7 — main performance figure.*

**Protocol:**
- Events: `simulate_controlled_family` with timing jitter so k = 2 effect is visible.
- For each noise type and each rank k = 1 … K_max, train EMPCA on the train split.
- Amplitude estimate = GLS coefficient for PC1 (re-scaled to match OF on train set).
- σ_E = std of amplitude estimates on the test split.
- Normalize by the OF CRB (1/√N_Φ) so all noise types start at 1.0 at k = 1.
- Error bands = std across N_seeds.

**Pass criterion:** σ_E(k=1) / CRB ∈ [0.98, 1.02] for all noise types.


In [ ]:
# ── P1: compute σ_E vs k across noise types and seeds ───────────────────────
from numpy.polynomial import polynomial as Poly

def estimate_sigma_E_vs_k(cfg, noise_type, n_events, noise_power, k_max, seed):
    """Return (sigma_E[k], crb) for k=1..k_max using controlled-family simulation."""
    _cfg = CanonicalConfig(**{**cfg.__class__.__dataclass_fields__ and
                             {f: getattr(cfg, f) for f in cfg.__dataclass_fields__},
                             'seed': seed,
                             'sim_events_large': n_events,
                             'sim_events_medium': n_events})

    bundle, truth = simulate_controlled_family(
        _cfg,
        n_events=n_events,
        tau_decay_range=(3e6, 3e6),          # fixed shape
        t0_jitter_range=(-5e4, 5e4),         # mild timing jitter
        n_qp_range=(5000, 5000),
        noise_type=noise_type,
        noise_power=noise_power,
        arrival_mode='aligned',
    )
    train_idx = bundle.split_indices['train']
    test_idx  = bundle.split_indices['test']
    w = bundle.weights_one_sided
    Xf_train  = bundle.traces_freq[train_idx]
    Xf_test   = bundle.traces_freq[test_idx]

    # CRB from template + weights
    of_stats = compute_of_statistics(bundle.template_freq, w)
    crb = float(of_stats['sigma_amplitude'])

    sigma_k = []
    for k in range(1, k_max + 1):
        exact = exact_weighted_subspace(Xf_train, w, k=k)
        basis = exact['basis_native']          # use exact SVD for speed
        coeff_test = rankk_gls_coefficients(Xf_test, basis, w, return_complex=False)
        # rescale first coefficient using train-set OF amplitude
        of_train = rankk_gls_coefficients(Xf_train, bundle.template_freq[None, :], w, return_complex=False).ravel()
        c1_train  = rankk_gls_coefficients(Xf_train, basis[:1], w, return_complex=False).ravel()
        # linear scale factor: a1 -> OF amplitude
        scale = float(np.dot(of_train, c1_train) / (np.dot(c1_train, c1_train) + 1e-30))
        amp_hat = coeff_test[:, 0] * scale
        sigma_k.append(float(np.std(amp_hat, ddof=1)))
    return np.array(sigma_k), crb


print('Running P1 simulations across seeds (this may take a few minutes)...')
p1_results = {nt: [] for nt in NOISE_TYPES}
p1_crb     = {nt: [] for nt in NOISE_TYPES}

for seed_offset in range(N_SEEDS):
    seed = cfg.seed + seed_offset * 7919
    for nt in NOISE_TYPES:
        sigma_k, crb = estimate_sigma_E_vs_k(cfg, nt, N_EVENTS, NOISE_POWER, K_MAX, seed)
        p1_results[nt].append(sigma_k)
        p1_crb[nt].append(crb)
        print(f'  seed={seed}  noise={nt}  sigma(k=1)={sigma_k[0]:.4f}  CRB={crb:.4f}')

print('Done.')


In [ ]:
# ── P1: plot ─────────────────────────────────────────────────────────────────
ks = np.arange(1, K_MAX + 1)
fig, ax = plt.subplots(figsize=(6.5, 4.5))

for nt in NOISE_TYPES:
    arr    = np.array(p1_results[nt])      # shape (N_seeds, K_max)
    crb_arr= np.array(p1_crb[nt])
    crb_mu = float(np.mean(crb_arr))

    mu_norm = arr.mean(0) / crb_mu
    sd_norm = arr.std(0)  / crb_mu

    ax.errorbar(ks, mu_norm, yerr=sd_norm,
                fmt='o-', color=NOISE_COLORS[nt], capsize=4,
                label=NOISE_LABELS[nt])

ax.axhline(1.0, color='gray', ls='--', lw=1.2, label='OF baseline (CRB)')
ax.set_xlabel('EMPCA rank k')
ax.set_ylabel(r'$\sigma_E / \sigma_E^{\mathrm{OF}}$  (normalised)')
ax.set_xticks(ks)
ax.set_title('P1 — Energy resolution vs rank k')
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, 'p1_sigma_vs_k.pdf')
plt.show()

# ── pass/fail table ───────────────────────────────────────────────────────────
rows = []
for nt in NOISE_TYPES:
    arr    = np.array(p1_results[nt])
    crb_arr= np.array(p1_crb[nt])
    ratio  = float(arr[:, 0].mean() / np.mean(crb_arr))
    rows.append({'noise_type': nt, 'sigma_E(k=1)/CRB': round(ratio, 4),
                 'pass [0.98,1.02]': 'PASS' if 0.98 <= ratio <= 1.02 else 'FAIL'})
display(pd.DataFrame(rows))


In [ ]:
# ── P1 verification: confirm EM-EMPCA matches exact SVD for seed 314159 ───────
_cfg_v = CanonicalConfig(seed=314159).validate()
bundle_v, _ = simulate_rank1_batch(_cfg_v, n_events=300,
                                    amp_range=(0.8, 1.2), noise_type='pink')
w_v = bundle_v.weights_one_sided
train_v = bundle_v.split_indices['train']
exact_v   = exact_weighted_subspace(bundle_v.traces_freq[train_v], w_v, k=1)
empca_v   = fit_weighted_empca(bundle_v.traces_freq[train_v], w_v, k=1, n_iter=50)
try:
    from empca_equivalence_utils import weighted_cosine as _wcos
except ImportError:
    _wcos = weighted_cosine
cos_v = abs(float(np.real(_wcos(exact_v['basis_native'][0],
                                 empca_v['basis'][0], w_v))))
print(f'Exact SVD vs EM-EMPCA cosine = {cos_v:.8f}  (expect > 0.9999)')
assert cos_v > 0.9999, 'EM-EMPCA diverges from SVD — check convergence'


---
## P2 — SNR Spectral Density |T(f)|²/J(f) with PC1 Overlay

*Paper §4.3 — visual proof of Theorem 1.*

For each noise type: plot J(f), |T(f)|², and |T(f)|²/J(f) together with the
learned |PC1(f)|² (normalized to the same scale). The near-proportionality of
PC1 to T/J is a visual proof that rank-1 EMPCA ≡ optimal filter.


In [ ]:
# ── P2: compute PSD, SNR density, and PC1 per noise type ─────────────────────
p2_data = {}

for nt in NOISE_TYPES:
    bundle, _ = simulate_rank1_batch(
        cfg, n_events=N_EVENTS,
        amp_range=(0.8, 1.2),
        noise_type=nt, noise_power=NOISE_POWER,
    )
    train_idx = bundle.split_indices['train']
    w = bundle.weights_one_sided

    # exact rank-1 subspace (= SVD-optimal, matches OF by Theorem 1)
    exact1 = exact_weighted_subspace(bundle.traces_freq[train_idx], w, k=1)
    pc1_f  = phase_align_basis(exact1['basis_native'][0], bundle.template_freq, w)

    n_freq = len(bundle.psd_one_sided)
    freq_axis = np.fft.rfftfreq(cfg.trace_len, d=1.0 / cfg.sampling_frequency)

    # |T(f)|^2 and SNR density
    Tf2    = np.abs(bundle.template_freq) ** 2
    Jf     = bundle.psd_one_sided.copy(); Jf[0] = np.nan
    snr_f  = np.where(Jf > 0, Tf2 / Jf, 0.0)
    pc1_mag2 = np.abs(pc1_f) ** 2

    # normalise PC1 magnitude to match SNR density at its own peak
    valid = snr_f > 0
    if valid.sum() > 0 and pc1_mag2[valid].sum() > 0:
        scale = snr_f[valid].sum() / pc1_mag2[valid].sum()
    else:
        scale = 1.0

    N_phi = float(np.nansum(snr_f))
    p2_data[nt] = {
        'freq_axis': freq_axis,
        'Jf': Jf,
        'Tf2': Tf2,
        'snr_f': snr_f,
        'pc1_mag2_scaled': pc1_mag2 * scale,
        'N_phi': N_phi,
        'crb_sigma': 1.0 / np.sqrt(max(N_phi, 1e-30)),
    }
    print(f'  {nt}: N_Phi={N_phi:.1f}  sigma_CRB={p2_data[nt]["crb_sigma"]:.4f}')


In [ ]:
# ── P2: plot (3 panels, one per noise type) ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)

for ax, nt in zip(axes, NOISE_TYPES):
    d = p2_data[nt]
    f = d['freq_axis'][1:]            # drop DC

    ax2 = ax.twinx()
    ax.semilogy(f, d['Jf'][1:],   color='gray', lw=1.2, ls='--', label='J(f)')
    ax.semilogy(f, d['Tf2'][1:],  color=RED,   lw=1.2, ls=':',  label='|T(f)|²')
    ax2.semilogy(f, d['snr_f'][1:],          color=BLUE,   lw=1.8,      label='|T|²/J  (SNR dens.)')
    ax2.semilogy(f, d['pc1_mag2_scaled'][1:], color=ORANGE, lw=1.4, ls='-.', label='|PC1(f)|²  (scaled)')

    ax.set_xlabel('frequency [Hz]')
    ax.set_ylabel('J(f) or |T(f)|²', color='gray')
    ax2.set_ylabel('SNR density / PC1', color=BLUE)
    valid_p2 = d['snr_f'][1:] > 0
    r_pearson = float(np.corrcoef(d['snr_f'][1:][valid_p2],
                                   d['pc1_mag2_scaled'][1:][valid_p2])[0, 1])
    ax.set_title(f'{NOISE_LABELS[nt]}\nN_Φ={d["N_phi"]:.0f}  σ_CRB={d["crb_sigma"]:.4f}  r={r_pearson:.4f}')
    ax.tick_params(axis='y', colors='gray')
    ax2.tick_params(axis='y', colors=BLUE)

    # combined legend
    lines1, lbl1 = ax.get_legend_handles_labels()
    lines2, lbl2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, lbl1 + lbl2, fontsize=7.5, loc='upper right')

fig.suptitle('P2 — SNR Spectral Density and Learned PC1  (Theorem 1 visual)', y=1.01)
fig.tight_layout()
save_fig(fig, 'p2_snr_spectral_density.pdf')
plt.show()


---
## P3 — 3 × 3 PC Cosine Score Matrix (PC × Noise Type)

*Paper §4 — shows PC interpretation is noise-agnostic.*

For each noise type, train rank-3 EMPCA on `simulate_controlled_family` (with
timing and shape variation).  Compute weighted cosine between each PCk and its
physically motivated reference direction.  Report as a heatmap with bootstrap CIs.

| PC | Reference direction |
|---|---|
| PC1 | Normalised mean frequency vector |
| PC2 | Derivative of mean pulse → frequency domain |
| PC3 | Width proxy t·s′(t) → frequency domain |

Pass criterion: |cos(PC1, mean)| > 0.95; |cos(PC2, deriv)| > 0.80.


In [ ]:
# ── P3: build reference directions ──────────────────────────────────────────
def build_references(bundle):
    """Return (ref_pc1, ref_pc2, ref_pc3) in frequency domain, weighted-normalised."""
    w    = bundle.weights_one_sided
    t    = bundle.template_time
    tf   = bundle.template_freq

    # PC1 reference: mean frequency direction (same as template for rank-1 signal)
    ref1 = normalize_weighted(np.fft.rfft(np.mean(bundle.traces_baseline, axis=0)), w)

    # PC2 reference: derivative of mean pulse
    deriv_t = np.gradient(np.mean(bundle.traces_baseline, axis=0))
    ref2 = normalize_weighted(np.fft.rfft(deriv_t), w)

    # PC3 reference: analytical d(s)/d(tau) for exponential template s(t) = exp(-t/tau)
    # d(s)/d(tau) = (t / tau^2) * exp(-t / tau)
    # This replaces the old heuristic  width_t = np.arange(n) * np.gradient(t)
    tau_nominal = 3e-3   # 3 ms nominal decay constant (adjust if cfg exposes this)
    t_sec = np.arange(len(t)) / cfg.sampling_frequency
    width_t = (t_sec / tau_nominal**2) * np.exp(-t_sec / tau_nominal)
    ref3 = normalize_weighted(np.fft.rfft(width_t), w)

    return ref1, ref2, ref3


In [ ]:
# ── P3: cosine scores with bootstrap CIs ─────────────────────────────────────
N_BOOT = 200    # bootstrap resamples; increase to 500 for final paper run

def cosine_scores_bootstrap(bundle, n_boot=200, seed=0):
    """Return dict: pc1/pc2/pc3 -> {'mean', 'ci_lo', 'ci_hi'}."""
    rng = np.random.default_rng(seed)
    w   = bundle.weights_one_sided
    ref1, ref2, ref3 = build_references(bundle)
    train_idx = bundle.split_indices['train']
    Xf_train  = bundle.traces_freq[train_idx]
    n_train   = len(train_idx)

    boot_cos = {f'pc{k}': [] for k in (1, 2, 3)}
    for _ in range(n_boot):
        idx = rng.integers(0, n_train, size=n_train)
        Xb  = Xf_train[idx]
        exact = exact_weighted_subspace(Xb, w, k=3)
        bases = exact['basis_native']
        # sign-align to reference
        for j, ref in enumerate((ref1, ref2, ref3)):
            pc = bases[j]
            if np.real(weighted_inner(pc, ref, w)) < 0:
                pc = -pc
            boot_cos[f'pc{j+1}'].append(abs(float(np.real(weighted_cosine(pc, ref, w)))))

    result = {}
    for key, vals in boot_cos.items():
        arr = np.array(vals)
        result[key] = {
            'mean':   float(np.mean(arr)),
            'ci_lo':  float(np.percentile(arr, 2.5)),
            'ci_hi':  float(np.percentile(arr, 97.5)),
        }
    return result


print('Running P3 bootstrap (may take ~1 min)...')
p3_scores = {}
for nt in NOISE_TYPES:
    bundle, _ = simulate_controlled_family(
        cfg, n_events=N_EVENTS,
        tau_decay_range=(3e6, 3e6),
        t0_jitter_range=(-5e4, 5e4),
        n_qp_range=(5000, 5000),
        noise_type=nt, noise_power=NOISE_POWER,
    )
    p3_scores[nt] = cosine_scores_bootstrap(bundle, n_boot=N_BOOT, seed=cfg.seed)
    print(f'  {nt}: PC1={p3_scores[nt]["pc1"]["mean"]:.3f}  PC2={p3_scores[nt]["pc2"]["mean"]:.3f}  PC3={p3_scores[nt]["pc3"]["mean"]:.3f}')
print('Done.')


In [ ]:
# ── P3: heatmap + CI plot ─────────────────────────────────────────────────────
pcs = ['pc1', 'pc2', 'pc3']
pc_labels = ['PC1 vs mean', 'PC2 vs deriv', 'PC3 vs width']
heat_mean = np.array([[p3_scores[nt][pc]['mean']   for nt in NOISE_TYPES] for pc in pcs])
heat_lo   = np.array([[p3_scores[nt][pc]['ci_lo']  for nt in NOISE_TYPES] for pc in pcs])
heat_hi   = np.array([[p3_scores[nt][pc]['ci_hi']  for nt in NOISE_TYPES] for pc in pcs])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={'width_ratios': [1.3, 2]})

# left: heatmap
im = axes[0].imshow(heat_mean, vmin=0, vmax=1, cmap='Blues', aspect='auto')
axes[0].set_xticks(range(len(NOISE_TYPES)))
axes[0].set_xticklabels([NOISE_LABELS[n] for n in NOISE_TYPES], rotation=20, ha='right', fontsize=9)
axes[0].set_yticks(range(3))
axes[0].set_yticklabels(pc_labels)
axes[0].set_title('P3 — Weighted cosine (mean over bootstrap)')
plt.colorbar(im, ax=axes[0])
for i in range(3):
    for j in range(len(NOISE_TYPES)):
        axes[0].text(j, i, f'{heat_mean[i, j]:.3f}', ha='center', va='center', fontsize=9,
                     color='white' if heat_mean[i, j] > 0.7 else 'black')

# right: CI error bar plot
x = np.arange(len(NOISE_TYPES))
width = 0.22
for pi, (pc, lbl) in enumerate(zip(pcs, pc_labels)):
    mu  = np.array([p3_scores[nt][pc]['mean']   for nt in NOISE_TYPES])
    lo  = np.array([p3_scores[nt][pc]['ci_lo']  for nt in NOISE_TYPES])
    hi  = np.array([p3_scores[nt][pc]['ci_hi']  for nt in NOISE_TYPES])
    xpos = x + (pi - 1) * width
    axes[1].errorbar(xpos, mu, yerr=[mu - lo, hi - mu],
                     fmt='o', capsize=5, label=lbl,
                     color=[BLUE, ORANGE, GREEN][pi])

axes[1].axhline(0.95, ls='--', lw=1, color='gray', label='pass threshold PC1 (0.95)')
axes[1].axhline(0.80, ls=':',  lw=1, color='gray', label='pass threshold PC2 (0.80)')
axes[1].set_xticks(x)
axes[1].set_xticklabels([NOISE_LABELS[n] for n in NOISE_TYPES], rotation=15)
axes[1].set_ylabel('|weighted cosine|')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('P3 — Bootstrap 95% CI per noise type')
axes[1].legend(fontsize=8)

fig.tight_layout()
save_fig(fig, 'p3_cosine_matrix.pdf')
plt.show()


---
## P4 — Timing-Bias Reduction: Rank-1 vs Rank-2

*Paper §4 — physical consequence of PC2 interpretation.*

Generate events with large timing jitter at **fixed true amplitude** = 1.
Estimate amplitude at rank 1 and rank 2 using GLS projection.
Show that rank-2 reduces amplitude bias introduced by timing jitter.

Pass criterion: |mean_bias(k=2)| < 0.7 × |mean_bias(k=1)|.


In [ ]:
# ── P4: simulate large-jitter events with fixed amplitude ────────────────────
def estimate_amplitude_rankk(bundle, k, train_idx, test_idx):
    """Return amplitude estimates on test set using rank-k GLS."""
    w        = bundle.weights_one_sided
    Xf_train = bundle.traces_freq[train_idx]
    Xf_test  = bundle.traces_freq[test_idx]

    exact = exact_weighted_subspace(Xf_train, w, k=k)
    basis = exact['basis_native']

    # train-set GLS coefficients
    coeff_train = rankk_gls_coefficients(Xf_train, basis, w, return_complex=False)

    # linear map from k coefficients -> amplitude (trained vs OF amplitude proxy)
    of_amp_train = rankk_gls_coefficients(
        Xf_train, bundle.template_freq[None, :], w, return_complex=False).ravel()

    beta, *_ = np.linalg.lstsq(coeff_train, of_amp_train, rcond=None)

    coeff_test = rankk_gls_coefficients(Xf_test, basis, w, return_complex=False)
    return coeff_test @ beta


p4_results = {}
for nt in NOISE_TYPES:
    bundle, truth = simulate_controlled_family(
        cfg, n_events=N_EVENTS,
        tau_decay_range=(3e6, 3e6),
        t0_jitter_range=(-2e5, 2e5),    # large timing jitter
        n_qp_range=(5000, 5000),
        noise_type=nt, noise_power=NOISE_POWER,
    )
    train_idx = bundle.split_indices['train']
    test_idx  = bundle.split_indices['test']

    # CORRECT: use simulation truth rather than OF amplitude as ground truth
    # (OF amplitude is circular: it is what we're trying to measure)
    if hasattr(truth, 'amplitudes'):
        true_amp_test = truth.amplitudes[test_idx]
    elif hasattr(truth, 'true_amplitudes'):
        true_amp_test = truth.true_amplitudes[test_idx]
    elif hasattr(truth, 'amp'):
        true_amp_test = truth.amp[test_idx]
    else:
        # Fallback: normalise n_qp_per_event to unit-amplitude scale
        raise AttributeError(
            'truth object has no amplitudes / true_amplitudes / amp attribute. '
            'Add to simulate_controlled_family: return bundle, '
            'SimpleNamespace(amplitudes=amp_array, ...)')

    # Normalise so that mean true amplitude = 1 on the training set
    true_amp_train = (truth.amplitudes if hasattr(truth, 'amplitudes')
                      else true_amp_test)[train_idx]
    amp_scale = float(np.mean(true_amp_train))

    amp_k1 = estimate_amplitude_rankk(bundle, 1, train_idx, test_idx) / amp_scale
    amp_k2 = estimate_amplitude_rankk(bundle, 2, train_idx, test_idx) / amp_scale
    true_amp_norm = true_amp_test / amp_scale

    p4_results[nt] = {
        'amp_k1':   amp_k1,
        'amp_k2':   amp_k2,
        'true_amp': true_amp_norm,
        # bias relative to simulation truth (not hardcoded 1.0)
        'bias_k1': float(np.mean(amp_k1 - true_amp_norm)),
        'bias_k2': float(np.mean(amp_k2 - true_amp_norm)),
        'rmse_k1': float(np.sqrt(np.mean((amp_k1 - true_amp_norm)**2))),
        'rmse_k2': float(np.sqrt(np.mean((amp_k2 - true_amp_norm)**2))),
    }
    r = p4_results[nt]
    print(f'{nt}: bias k1={r["bias_k1"]:.4f}  k2={r["bias_k2"]:.4f}  '
          f'RMSE k1={r["rmse_k1"]:.4f}  k2={r["rmse_k2"]:.4f}')


In [ ]:
# ── P4: plot amplitude distributions + summary table ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)

for ax, nt in zip(axes, NOISE_TYPES):
    r = p4_results[nt]
    bins = np.linspace(
        min(r['amp_k1'].min(), r['amp_k2'].min()) - 0.02,
        max(r['amp_k1'].max(), r['amp_k2'].max()) + 0.02,
        40)
    ax.hist(r['amp_k1'], bins=bins, alpha=0.55, color=BLUE,   label=f'rank-1  bias={r["bias_k1"]:+.3f}')
    ax.hist(r['amp_k2'], bins=bins, alpha=0.55, color=ORANGE, label=f'rank-2  bias={r["bias_k2"]:+.3f}')
    ax.axvline(1.0, color='gray', ls='--', lw=1)
    ax.set_title(f'{NOISE_LABELS[nt]}')
    ax.set_xlabel('estimated amplitude (normalised)')
    ax.set_ylabel('count')
    ax.legend(fontsize=8)

fig.suptitle('P4 — Amplitude distributions: rank-1 vs rank-2  (fixed true amplitude, large timing jitter)')
fig.tight_layout()
save_fig(fig, 'p4_timing_bias_reduction.pdf')
plt.show()

summary_rows = []
for nt in NOISE_TYPES:
    r = p4_results[nt]
    bias_reduction = 1.0 - abs(r['bias_k2']) / max(abs(r['bias_k1']), 1e-12)
    summary_rows.append({
        'noise_type': nt,
        'bias_k1': round(r['bias_k1'], 5),
        'bias_k2': round(r['bias_k2'], 5),
        'rmse_k1': round(r['rmse_k1'], 5),
        'rmse_k2': round(r['rmse_k2'], 5),
        'bias_reduction_%': round(bias_reduction * 100, 1),
        'pass [>30%]': 'PASS' if bias_reduction > 0.30 else 'FAIL',
    })
display(pd.DataFrame(summary_rows))


---
## P5 — Residual χ² Distribution as a Function of k

*Paper §4.4 — rank saturation and goodness-of-fit.*

After projecting test events onto k PCs, the weighted residual per event
(divided by the mean residual) should follow χ²(1) under the null if all
signal variance is captured.  A QQ-plot and KS p-value for k = 1 … 4 reveal
when additional PCs stop capturing signal and start fitting noise.

KS p < 0.05 at k = 1 (underfitting for multi-D families) → rising p as k
increases → plateau near 1.0 at rank saturation.


In [ ]:
# ── P5: residual distributions vs k ─────────────────────────────────────────
K_PLOT = 4   # show k = 1..4

p5_results = {}
for nt in NOISE_TYPES:
    bundle, _ = simulate_controlled_family(
        cfg, n_events=N_EVENTS,
        tau_decay_range=(3e6, 3e6),
        t0_jitter_range=(-5e4, 5e4),
        n_qp_range=(5000, 5000),
        noise_type=nt, noise_power=NOISE_POWER,
    )
    train_idx = bundle.split_indices['train']
    test_idx  = bundle.split_indices['test']
    w = bundle.weights_one_sided
    Xf_train = bundle.traces_freq[train_idx]
    Xf_test  = bundle.traces_freq[test_idx]

    n_freq_bins = Xf_test.shape[1]   # number of frequency bins used

    p5_results[nt] = {}
    exact_all = exact_weighted_subspace(Xf_train, w, k=K_PLOT)
    for k in range(1, K_PLOT + 1):
        basis  = exact_all['basis_native'][:k]
        coeff  = rankk_gls_coefficients(Xf_test, basis, w, return_complex=True)
        resid  = residual_energy_per_trace(Xf_test, basis, coeff, w)

        # CORRECT null distribution: weighted residual under Gaussian data with
        # d frequency bins and rank-k projection follows chi2(d - k) at the
        # population level.  Scale so observed residuals match chi2(df_null).
        df_null = n_freq_bins - k
        resid_scaled = resid * df_null / resid.mean()

        # KS test against chi2(d - k)  [was wrongly tested against Exp(1)]
        ks_stat, ks_p = stats.kstest(
            resid_scaled,
            lambda x, df=df_null: stats.chi2.cdf(x, df)
        )
        p5_results[nt][k] = {
            'resid': resid,
            'resid_scaled': resid_scaled,
            'df_null': df_null,
            'ks_stat': ks_stat,
            'ks_p': ks_p,
        }

print('P5 computed.')


In [ ]:
# ── P5: QQ-plots + KS p-value panel ─────────────────────────────────────────
fig, axes = plt.subplots(len(NOISE_TYPES), K_PLOT,
                         figsize=(13, 3.5 * len(NOISE_TYPES)), sharey=False)
if len(NOISE_TYPES) == 1:
    axes = axes[None, :]

for ni, nt in enumerate(NOISE_TYPES):
    for k in range(1, K_PLOT + 1):
        ax  = axes[ni, k - 1]
        r   = p5_results[nt][k]
        df_null = r['df_null']
        resid_scaled = r['resid_scaled']

        # QQ against chi2(d - k)  [previously wrongly plotted vs Exp(1)]
        osm, osr = stats.probplot(resid_scaled,
                                   dist=stats.chi2(df_null), fit=False)
        ax.scatter(osm, osr, s=6, alpha=0.5, color=NOISE_COLORS[nt])
        lo = min(osm.min(), osr.min()); hi = max(osm.max(), osr.max())
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
        ax.set_title(f'{NOISE_LABELS[nt]}  k={k}\nKS p={r["ks_p"]:.3f}', fontsize=9)
        if k == 1:
            ax.set_ylabel('Sample quantiles')
        if ni == len(NOISE_TYPES) - 1:
            ax.set_xlabel(f'chi2({df_null}) quantiles')

fig.suptitle('P5 — Residual QQ-plots vs chi2(d-k) null  (k = 1 → 4)', y=1.01)
fig.tight_layout()
save_fig(fig, 'p5_residual_qq.pdf')
plt.show()

# KS p-value vs k summary
fig2, ax2 = plt.subplots(figsize=(6, 3.8))
for nt in NOISE_TYPES:
    ks_ps = [p5_results[nt][k]['ks_p'] for k in range(1, K_PLOT + 1)]
    ax2.plot(range(1, K_PLOT + 1), ks_ps, 'o-', color=NOISE_COLORS[nt],
             label=NOISE_LABELS[nt])
ax2.axhline(0.05, color='gray', ls='--', lw=1, label='alpha = 0.05')
ax2.set_xlabel('EMPCA rank k')
ax2.set_ylabel('KS p-value  (residual vs chi2(d-k))')
ax2.set_title('P5 — Goodness-of-fit vs rank  [null: chi2(d-k)]')
ax2.legend(fontsize=9)
fig2.tight_layout()
save_fig(fig2, 'p5_ks_pvalue_vs_k.pdf')
plt.show()


---
## P6 — Bootstrap CIs on Cosine Scores *(merged into P3)*

**Note:** P6 has been merged into P3.  The bootstrap CI figure (formerly
`p6_bootstrap_ci_cosines.pdf`) is now the right-hand panel of `p3_cosine_matrix.pdf`,
referenced in the paper as part of Figure `fig:p3-cosine`.

The cell below reproduces the standalone P6 figure for completeness, using the
bootstrap results already computed in P3.


In [ ]:
# Note: bootstrap CI figure (formerly P6) is the right-hand panel of
# p3_cosine_matrix.pdf.  It is referenced in the paper as part of Figure
# fig:p3-cosine.  This cell is kept for completeness only.
# P6 experiment is considered merged into P3.

# ── P6: bootstrap CI summary figure ──────────────────────────────────────────
# Use the per-noise-type bootstrap from P3.
# For the paper we show results averaged across noise types, with per-type breakdown.

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(3)
width_bar = 0.22

for ni, nt in enumerate(NOISE_TYPES):
    means = [p3_scores[nt][f'pc{k}']['mean']   for k in (1, 2, 3)]
    lo    = [p3_scores[nt][f'pc{k}']['ci_lo']  for k in (1, 2, 3)]
    hi    = [p3_scores[nt][f'pc{k}']['ci_hi']  for k in (1, 2, 3)]
    xpos  = x + (ni - 1) * width_bar
    ax.errorbar(xpos, means,
                yerr=[np.array(means) - np.array(lo), np.array(hi) - np.array(means)],
                fmt='o', capsize=6, ms=6,
                color=NOISE_COLORS[nt], label=NOISE_LABELS[nt])

ax.axhline(0.95, ls='--', lw=1, color='gray', label='PC1 pass threshold')
ax.axhline(0.80, ls=':',  lw=1, color='gray', label='PC2 pass threshold')
ax.set_xticks(x)
ax.set_xticklabels(['PC1 vs mean', 'PC2 vs deriv', 'PC3 vs width'])
ax.set_ylabel('|weighted cosine|  (bootstrap mean ± 95% CI)')
ax.set_ylim(0, 1.08)
ax.set_title('P6 — Bootstrap 95% CIs on PC cosine scores')
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, 'p6_bootstrap_ci_cosines.pdf')
plt.show()

# Numeric summary
rows = []
for nt in NOISE_TYPES:
    for k in (1, 2, 3):
        s = p3_scores[nt][f'pc{k}']
        ci_width = s['ci_hi'] - s['ci_lo']
        rows.append({
            'noise_type': nt,
            'PC': f'PC{k}',
            'mean': round(s['mean'], 4),
            'ci_lo': round(s['ci_lo'], 4),
            'ci_hi': round(s['ci_hi'], 4),
            'CI_width': round(ci_width, 4),
            'stable [<0.05]': 'YES' if ci_width < 0.05 else 'NO',
        })
display(pd.DataFrame(rows))


---
## Figure Summary

| File | Experiment | Paper section |
|---|---|---|
| `p1_sigma_vs_k.pdf` | P1: σ_E vs rank k, three noise types | §7 main results |
| `p2_snr_spectral_density.pdf` | P2: SNR density + PC1 overlay | §4.3 Theorem 1 visual |
| `p3_cosine_matrix.pdf` | P3: 3×3 cosine heatmap + CI bars | §4 PC interpretation |
| `p4_timing_bias_reduction.pdf` | P4: amplitude distributions k=1 vs k=2 | §4 physical consequence |
| `p5_residual_qq.pdf` | P5: residual QQ-plots vs k | §4.4 rank saturation |
| `p5_ks_pvalue_vs_k.pdf` | P5: KS p-value vs rank | §4.4 rank saturation |
| `p6_bootstrap_ci_cosines.pdf` | P6: bootstrap 95% CIs on cosine scores | §4 PC interpretation |


In [ ]:
print('All figures in:', FIG_DIR)
for f in sorted(FIG_DIR.glob('p*.pdf')):
    print(' ', f.name)
